# Phase 1: Exploratory Data Analysis (EDA) on Bronze Raw Layer

## Overview
This initial profiling step inspects the raw, flattened string payload stored in `cr_seismic_analysis.bronze.seismic_raw`. Analyzing Bronze allows us to understand data quality issues, unparsed timestamps, and spatial boundaries *before* designing the Silver cleansing logic.

### Objectives:
1. **Raw Volume Audit:** Verify total ingested records from the raw JSON payload.
2. **Raw Schema Inspection:** Review initial column definitions and confirm string-based data types.
3. **Data Sampling:** Inspect the first 10 raw records to identify necessary type conversions, formatting fixes, and filtering rules needed for Silver.

In [0]:
%sql

-- Determine the amount of rows in the bronze table

SELECT COUNT(*) FROM cr_seismic_analysis.bronze.seismic_raw;

In [0]:
%sql
-- Explore data structure - What columns? What types of data? 
DESCRIBE TABLE cr_seismic_analysis.bronze.seismic_raw;

In [0]:
%sql
SELECT 
    event_id,
    magnitude,
    magnitude_type,
    place,
    event_timestamp_raw,
    event_type,
    longitude,
    latitude,
    depth,
    source_file,
    ingestion_timestamp
FROM 
    cr_seismic_analysis.bronze.seismic_raw
LIMIT 10;

# Phase 2: Null Value Profiling & Data Completeness Analysis

## Overview
To establish data quality baselines for the **Silver layer**, we conduct a completeness audit on `cr_seismic_analysis.bronze.seismic_raw`. 

This step evaluates missingness across two perspectives:
1. **Absolute Null Counts:** Exact number of missing values per field.
2. **Relative Missingness Percentages (%):** Proportional impact of missing data across the entire dataset.

### Key Findings & Quality Takeaways

* **High Data Completeness:** None of the core attributes (`event_id`, `magnitude`, `latitude`, `longitude`, `event_timestamp_raw`) exhibit critical missingness rates ($> 50\%$).
* **Pipeline Impact:** Because the raw Bronze layer contains negligible null values across key fields, complex missing-value imputation algorithms or aggressive `DROP` rules are **unnecessary** during the Bronze $\rightarrow$ Silver transition.
* **Silver Strategy:** Standard `WHERE` filters will suffice to catch any rare anomalies, keeping our transformation pipeline clean, deterministic, and fast.

In [0]:
%sql
WITH total AS (
    SELECT 
        COUNT(*) AS total_events 
    FROM cr_seismic_analysis.bronze.seismic_raw
),
null_data AS (
    SELECT
        COUNT(*) - COUNT(event_id) AS nulls_id,
        COUNT(*) - COUNT(magnitude) AS nulls_magnitudes,
        COUNT(*) - COUNT(magnitude_type) AS nulls_magnitude_types,
        COUNT(*) - COUNT(place) AS nulls_places,
        COUNT(*) - COUNT(event_timestamp_raw) AS nulls_ets,
        COUNT(*) - COUNT(event_type) AS nulls_events,
        COUNT(*) - COUNT(longitude) AS nulls_longitudes,
        COUNT(*) - COUNT(latitude) AS nulls_latitudes,
        COUNT(*) - COUNT(depth) AS nulls_depths,
        COUNT(*) - COUNT(source_file) AS nulls_sources,
        COUNT(*) - COUNT(ingestion_timestamp) AS nulls_ingest

    FROM cr_seismic_analysis.bronze.seismic_raw
)
SELECT 
    t.total_events,
    n.*
FROM total t, null_data n


In [0]:
%sql
WITH total_count AS (
    SELECT COUNT(*) AS total FROM cr_seismic_analysis.bronze.seismic_raw
)
SELECT 
    t.total AS total_events,
    ROUND((t.total - COUNT(event_id)) * 100.0 / t.total, 2) AS pct_null_id,
    ROUND((t.total - COUNT(magnitude)) * 100.0 / t.total, 2) AS pct_null_magnitude,
    ROUND((t.total - COUNT(magnitude_type)) * 100.0 / t.total, 2) AS pct_null_magnitude_type,
    ROUND((t.total - COUNT(place)) * 100.0 / t.total, 2) AS pct_null_place,
    ROUND((t.total - COUNT(event_timestamp_raw)) * 100.0 / t.total, 2) AS pct_null_ets,
    ROUND((t.total - COUNT(event_type)) * 100.0 / t.total, 2) AS pct_null_event,
    ROUND((t.total - COUNT(longitude)) * 100.0 / t.total, 2) AS pct_null_longitude,
    ROUND((t.total - COUNT(latitude)) * 100.0 / t.total, 2) AS pct_null_latitude,
    ROUND((t.total - COUNT(depth)) * 100.0 / t.total, 2) AS pct_null_depth,
    ROUND((t.total - COUNT(source_file)) * 100.0 / t.total, 2) AS pct_null_source,
    ROUND((t.total - COUNT(ingestion_timestamp)) * 100.0 / t.total, 2) AS pct_null_ingest
FROM cr_seismic_analysis.bronze.seismic_raw
CROSS JOIN total_count t
GROUP BY t.total;

# Phase 3: Key Takeaways 

* **Magnitude Scales:** Identified the dominant magnitude measurement types (`md`, `ml`, `mw`, etc.) to inform categorization rules.
* **Spatial Integrity Check:** Evaluated min/max coordinate ranges to detect corrupted values, coordinate flips, or global events outside the target boundary.
* **Silver Action Item:** Enforce strict bounding box criteria (`latitude BETWEEN 8.0 AND 11.5` and `longitude BETWEEN -86.0 AND -82.5`) during Silver transformation to isolate Costa Rican seismic activity.

In [0]:
%sql
-- High-level cardinality overview across key attributes
SELECT 
    COUNT(DISTINCT event_id) AS unique_events,
    COUNT(DISTINCT place) AS unique_places,
    COUNT(DISTINCT magnitude_type) AS unique_mag_types,
    COUNT(DISTINCT event_type) AS unique_event_types,
    COUNT(DISTINCT source_file) AS unique_source_files
FROM cr_seismic_analysis.bronze.seismic_raw;

In [0]:
%sql 
SELECT 
    magnitude,
    COUNT(*) AS quantity,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS percentage
FROM cr_seismic_analysis.bronze.seismic_raw
GROUP BY magnitude
ORDER BY quantity DESC;

In [0]:
%sql 
SELECT 
    magnitude_type,
    COUNT(*) AS quantity,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS percentage
FROM cr_seismic_analysis.bronze.seismic_raw
GROUP BY magnitude_type
ORDER BY quantity DESC;

In [0]:
%sql
-- Statistical boundary checks for numeric fields (stored as strings in Bronze)
SELECT 
    MIN(CAST(magnitude AS DOUBLE)) AS min_magnitude,
    MAX(CAST(magnitude AS DOUBLE)) AS max_magnitude,
    ROUND(AVG(CAST(magnitude AS DOUBLE)), 2) AS avg_magnitude,
    MIN(CAST(depth AS DOUBLE)) AS min_depth_km,
    MAX(CAST(depth AS DOUBLE)) AS max_depth_km,
    ROUND(AVG(CAST(depth AS DOUBLE)), 2) AS avg_depth_km,
    MIN(CAST(latitude AS DOUBLE)) AS min_latitude,
    MAX(CAST(latitude AS DOUBLE)) AS max_latitude,
    MIN(CAST(longitude AS DOUBLE)) AS min_longitude,
    MAX(CAST(longitude AS DOUBLE)) AS max_longitude
FROM cr_seismic_analysis.bronze.seismic_raw;

# Phase 4: Descriptive Statistics of Numerical Variables

In [0]:
%sql
-- Percentile profiling to detect skewness and non-normal distributions
SELECT 
    ROUND(percentile_approx(CAST(magnitude AS DOUBLE), 0.05), 2) AS mag_p05,
    ROUND(percentile_approx(CAST(magnitude AS DOUBLE), 0.25), 2) AS mag_p25,
    ROUND(percentile_approx(CAST(magnitude AS DOUBLE), 0.50), 2) AS mag_median,
    ROUND(percentile_approx(CAST(magnitude AS DOUBLE), 0.75), 2) AS mag_p75,
    ROUND(percentile_approx(CAST(magnitude AS DOUBLE), 0.95), 2) AS mag_p95,
    
    ROUND(percentile_approx(CAST(depth AS DOUBLE), 0.05), 2) AS depth_p05,
    ROUND(percentile_approx(CAST(depth AS DOUBLE), 0.25), 2) AS depth_p25,
    ROUND(percentile_approx(CAST(depth AS DOUBLE), 0.50), 2) AS depth_median,
    ROUND(percentile_approx(CAST(depth AS DOUBLE), 0.75), 2) AS depth_p75,
    ROUND(percentile_approx(CAST(depth AS DOUBLE), 0.95), 2) AS depth_p95
FROM cr_seismic_analysis.bronze.seismic_raw;

In [0]:
%sql
-- Evaluate Hypothesis 2: Micro/minor events (< 4.0) accounting for >= 85% of volume
SELECT 
    CASE 
        WHEN CAST(magnitude AS DOUBLE) < 4.0 THEN 'Minor Events (< 4.0)'
        ELSE 'Moderate/Strong Events (>= 4.0)'
    END AS magnitude_class,
    COUNT(*) AS total_events,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS percentage
FROM cr_seismic_analysis.bronze.seismic_raw
GROUP BY 1;

In [0]:
%sql
-- Distribution and average magnitude broken down by measurement scale
SELECT 
    magnitude_type,
    COUNT(*) AS total_records,
    ROUND(AVG(CAST(magnitude AS DOUBLE)), 2) AS avg_magnitude,
    MIN(CAST(magnitude AS DOUBLE)) AS min_magnitude,
    MAX(CAST(magnitude AS DOUBLE)) AS max_magnitude,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS pct_of_catalog
FROM cr_seismic_analysis.bronze.seismic_raw
GROUP BY magnitude_type
ORDER BY total_records DESC;